# Invocations API: Magentic Business Continuity Drill Job

This notebook mirrors `review_bot.scenarios.magentic_business_continuity_drill` and teaches the **magentic** orchestration pattern with code-defined agents. Scenario id: `magentic-business-continuity-drill`.

## 1. Notebook Setup

Run from the project virtual environment after `python -m pip install -e .`. The setup cell also applies the Aptos notebook theme (it falls back to a system sans-serif if Aptos is not installed).

In [ ]:
from pathlib import Path
import sys

def find_project_root(start):
    current = Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "review_bot").exists():
            return candidate
        nested = candidate / "invocations-api-review-bot"
        if (nested / "src" / "review_bot").exists():
            return nested
    raise RuntimeError("Could not find invocations-api-review-bot project root.")

PROJECT_ROOT = find_project_root(Path.cwd())
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

PROJECT_ROOT

## 2. Coded Agents, Not Prompt Agents

Every agent in this scenario is a **coded agent**. Two things make that true, and neither relies on a bare instructions prompt:

1. **Code-defined function tools.** Each agent is given real Python callables (from `code_tools.py`) such as `note_observation`, `rate_risk`, `make_checklist`, `extract_action_items`, `tally_votes`, and `compose_summary`. The model invokes this deterministic code to do its work.
2. **Code-defined orchestration.** The agents are wired together by custom `Executor` subclasses inside a `WorkflowBuilder` graph (or a code-driven orchestration builder), so the control flow, routing, aggregation, and state handling are all defined in code.

This mirrors the advanced Microsoft Agent Framework samples, where behaviour comes from executors, typed handlers, and function tools rather than prompt text alone.

## 3. Pattern Concept: Magentic

**Magentic orchestration** puts a manager agent in charge: it plans, delegates to specialists, and replans as new information arrives, stopping when the goal is met. Use it for open-ended work that needs dynamic planning.

## 4. API Fit: Responses vs Invocations

The orchestration is identical across both sample projects; only the API boundary differs.

- **Invocations API (this project):** the client sends a structured job payload and chooses the scenario per request; the handler returns an application-owned JSON result from `/invocations`.
- **Responses API (this project):** the client sends an OpenAI-style `input`, the scenario is fixed at server startup, and the coded workflow runs server-side behind a stable `/responses` contract.

Choose Responses for chat-style clients that expect a stable OpenAI-compatible shape; choose Invocations for systems that route different job types to different scenarios.

## 5. Load The Scenario And Helpers

Each scenario is a single `SCENARIO` object. Helper functions keep the notebook cells short and readable (PEP8-friendly).

In [ ]:
from review_bot.scenarios.magentic_business_continuity_drill import SCENARIO
from review_bot.scenarios import SCENARIOS_BY_ID, get_scenario
from review_bot.notebook_helpers import (
    agent_roster,
    apply_notebook_style,
    coded_agent_tool_map,
    default_ollama_options,
    invocation_reference,
    load_sample_payload,
    mcp_tool_context,
    pattern_anatomy,
    scenario_summary,
)

apply_notebook_style()
scenario = SCENARIO
assert SCENARIOS_BY_ID["magentic-business-continuity-drill"] is scenario
assert get_scenario("magentic-business-continuity-drill") is scenario
scenario_summary(scenario)

## 6. Load And Validate The Invocation Payload

Invocations owns its JSON contract. This is the same sample payload you can POST to `/invocations`.

In [ ]:
from review_bot.models import build_review_prompt, parse_review_request

payload = load_sample_payload(PROJECT_ROOT, scenario)
request = parse_review_request(payload)
assert request.scenario == scenario.id
payload

## 7. Pattern Anatomy

The framework-level behaviour of this orchestration pattern.

In [ ]:
pattern_anatomy(scenario)

## 8. Flow Diagram

A runtime Mermaid diagram of the orchestration. Dashed links show each coded agent's code tools (and MCP tools where used). The helper renders an image and returns the Mermaid source.

In [ ]:
from review_bot.diagram_helpers import display_scenario_flow

flow = display_scenario_flow(scenario)
flow.mermaid

## 9. MCP Tool Context

These agents are also grounded by the local `enterprise_context` MCP server (FastMCP over stdio; no network, credentials, or writes). Below are the tools it exposes and one deterministic sample call.

In [ ]:
from review_bot.mcp_servers import enterprise_context

context = mcp_tool_context(scenario)
print("Server exposes:", enterprise_context.AVAILABLE_TOOLS)
print("Tools used in this scenario:", context["all_tools_used"])
enterprise_context.search_policy("security review")

## 10. Agent Roster And Coded Tool Map

The roster lists each agent; the tool map shows the code tools (and any MCP tools) every coded agent can call. No agent is prompt-only.

In [ ]:
{
    "roster": agent_roster(scenario),
    "tool_map": coded_agent_tool_map(scenario),
}

## 11. Advanced Orchestration Internals

This scenario uses the framework's code-driven `MagenticBuilder`: a manager agent plans and delegates to the specialist participants, the framework maintains a progress ledger, and code-set round/stall/reset limits bound the run. The orchestration policy lives in code, not in a prompt.

## 12. Build The Prompt

The custom payload becomes the orchestration prompt.

In [ ]:
prompt = build_review_prompt(request)
print(prompt)

## 13. Live In-Process Run

This calls Ollama through the same `run_review` path as the handler.

In [ ]:
from review_bot.workflows import default_sample_max_tokens, run_review

max_tokens = default_sample_max_tokens(scenario)  # 500 for Magentic, 250 otherwise
response = await run_review(request, **default_ollama_options(), max_tokens=max_tokens)
response.to_dict()

## 14. API Boundary Reference

Compare the in-process workflow with the hosted `/invocations` endpoint.

In [ ]:
invocation_reference(scenario, request)

## 15. Experiments

Try changing an agent's `code_tools`, adjusting `max_tokens` or `temperature`, or editing a role brief in the scenario module, then rebuild and compare. You can also swap the orchestration pattern to see how the same coded agents behave under a different graph.